# 11 · Turn a failure into a learning signal

**Relay failure → Gym verifier → fresh attempts → RL update → replay**

We inspect notebook 09’s injected failure and train a tiny policy to choose better tool plans. This is a real CPU optimization exercise with **12 plan-selection parameters**, not Lightning fine-tuning.

**Who does what:** Nano coordinates the lesson tools. Relay supplies failure evidence, Gym scores fresh plans, and NeMo RL loss functions update the tiny policy. Lightning and its adapter remain unchanged.

###  verification, rewards, and policy updates

Gym **0.6.0** [environment tutorials](https://docs.nvidia.com/nemo/gym/v0.6.0/environment-tutorials/), [NeMo RL/Gym integration](https://docs.nvidia.com/nemo/rl/nightly/design-docs/nemo-gym-integration.html), and our [CPU experiment guide](../run/workshop/gym/README.md). RL's nightly documentation describes the full stack; this lesson uses only the functions pinned in [vendor/manifest.json](../run/workshop/gym/vendor/manifest.json).

retain 09's successful run and injected failure, set up the separate Gym environment, then run trace → case → learn. [learning.py](agent/learning.py) checks the failure evidence and launches CPU training; [resources.py](../run/workshop/gym/resources.py) verifies plans; [train_demo.py](../run/workshop/gym/train_demo.py) samples actions and updates weights.

**What changes:** 12 logits choose among four fixed plans for three intents. Gym evaluates fresh attempts through an in-process HTTP client; PyTorch applies 40 updates using selected upstream RL functions. The failure is manually mapped to an existing verifier rule, not ingested as training tensors. Lightning, OpenClaw, and the agent loop from 09 are not retrained or redeployed. Evaluation reuses this tiny environment.

The agent uses a workshop-authored training skill. See [Skills and training](README.md#skills-and-training) for the distinction between these instructions, NeMo libraries, and the separate NeMo Skills pipeline.


### Start here
Run the short cells in order. Helpers execute the existing lesson code and keep detailed logs expandable. All displayed results come from your run. Starting again creates a fresh output folder.

In [ ]:
from pathlib import Path
import sys, json
from IPython.display import JSON, display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "course/common.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from course.agent.lesson_walkthrough import LessonWalkthrough, table, bars, loss_chart, show_skill_library
lesson = LessonWalkthrough('11-trace-and-improve')

### 1 · Read the saved trace

This cell loads the manual run from notebook 09. Its outcome may be COMPLETE or HOLD. Keep observed failures separate from our intentionally injected test.

In [ ]:
lesson.step("trace")
record = lesson.ns["record"]
table(["Step", "Action", "Error"], [(e["step"], (e.get("action") or {}).get("tool"), e.get("error","")) for e in record["events"]])
print("Contract failures:", lesson.ns["check_trace"](record))

### 2 · Define a regression case

Deliberately inject **UNKNOWN sensor quality + CONFIRMED from one source**. The expected behavior is HOLD without a draft. The final check executes this regression separately from the missing-evidence learning exercise.

In [ ]:
lesson.step("case")
display(JSON(lesson.ns["case"], expanded=True))

### 3 · Run Gym and RL

| Plan | Reward rule |
|---|---|
| triage → draft | 0: evidence missing |
| perceive → triage → draft → review | 1: valid plan |

The old trace points to an existing Gym rule. It is not ingested as training tensors. The tool samples fresh plans and runs **40 CPU optimizer updates** using NeMo RL loss functions.

In [ ]:
lesson.step("learn")
learning = lesson.ns["learning_result"]
table(["Measurement", "Actual result"], [("Fresh attempts", learning["fresh_rollouts"]), ("Optimizer steps", learning["optimizer_steps"]), ("Weights changed", learning["weights_changed"]), ("Rejected plan reward", learning["failed_plan_reward"])])

### 4 · Compare and replay

These probabilities describe the small training environment—not performance on unseen tasks. Inspect the replayed plan and its fresh Gym result.

In [ ]:
bars({"Correct plan before": learning["before_correct_plan_probability"], "Correct plan after": learning["after_correct_plan_probability"], "Skip evidence before": learning["before_skip_evidence_probability"], "Skip evidence after": learning["after_skip_evidence_probability"]}, maximum=1)
print("Chosen plan:", learning["replayed_plan"])
table(["Tool", "Allowed"], [(s["tool"],s["allowed"]) for s in learning["replay_transitions"]])
print("Fresh Gym reward:", learning["replay_reward"])

### Check and save this manual run
Validate the artifacts and record this run for the next manual walkthrough. A successful check describes execution and evidence; read the actual quality scores separately.

In [ ]:
manual_result = lesson.finish()
display(JSON(manual_result, expanded=False))

### Grow the skill library
The library now includes this lesson’s skill, **workshop-review-trace**. Open the actual instructions below.

**`inspect-trace` → `test-regression` → `check-regression`**

In [ ]:
show_skill_library('11-trace-and-improve')

### Prepare OpenClaw
Run once for each new demo. This cell installs the growing skill library in a fresh sandbox workspace and starts a background host worker for this lesson’s three actions. **It does not ask the agent to do anything yet.**

Here, **installing a skill means placing its `SKILL.md` instruction file in OpenClaw’s workspace**. The current lesson’s source is `course/agent/skills/workshop-review-trace/SKILL.md` in this repository.

Each run gets this structure inside the OpenShell sandbox (launcher files omitted):

```text
/home/sandbox/course-runs/openclaw-<run-id>/
├── openclaw.json                 # points OpenClaw at this run’s workspace
├── chapter_tool.py              # helper invoked by the agent
├── chapter.json                 # this lesson’s task and allowed actions
└── workspace/
    ├── AGENTS.md                 # workspace instructions
    └── skills/
        ├── workshop-review-trace/
        │   └── SKILL.md
        └── ...                  # skills from earlier notebooks
```

OpenClaw discovers skills under `workspace/skills/`. The task you paste in the next step tells the agent which skill to read and supplies the full path to `chapter_tool.py` (the skill’s `<tool_path>` placeholder).

The agent invokes the helper through its execution tool. The helper forwards the selected action to the host worker, which runs the fixed lesson code and returns the result. Nano follows the skill and coordinates the calls.

Complete these earlier **OpenClaw demos** first: `08-agent-skills`, `09-agent-loop`. This run copies their latest verified agent artifacts; its inspect action reports the inputs.

Start the TUI task within **15 minutes** and keep this run’s `terminal_run` value for collecting results. If the worker expires, prepare a new run. The local run directory stores host results; the installed skills live at the sandbox paths printed below.


In [ ]:
from course.agent.chapter_terminal import prepare_chapter_terminal, collect_chapter_terminal
from course.agent.notebook_terminal import open_terminal
terminal_run = prepare_chapter_terminal('11-trace-and-improve')
print("Local run directory:", terminal_run["directory"])
print("OpenClaw sandbox workspace:", terminal_run["remote"] + "/workspace")
print("Installed skill:", terminal_run["remote"] + "/workspace/skills/workshop-review-trace/SKILL.md")
print("Python helper:", terminal_run["remote"] + "/chapter_tool.py")
print("\nThe agent will run these commands individually:")
for action in ['inspect-trace', 'test-regression', 'check-regression']:
    print("python3 " + terminal_run["remote"] + "/chapter_tool.py " + action)

### Open the TUI and paste the task
Exit the previous TUI with `/exit` first. This cell opens a terminal on the workshop computer’s desktop. Copy only the generated task into OpenClaw.

The terminal label is **Nano**, its lightweight orchestration model. Nano coordinates the lesson tools. Relay supplies failure evidence, Gym scores fresh plans, and NeMo RL loss functions update the tiny policy. Lightning and its adapter remain unchanged.

In [ ]:
open_terminal(terminal_run)
print("\nPASTE THIS INTO OPENCLAW:\n")
print(terminal_run["prompt"])

### Bring the evidence back
After the TUI finishes, run this cell. It retrieves that session’s trace and checks the artifacts and skill reads. If Run All reaches this cell before you finish the TUI task, run it again afterward.

**Look for:** Missing evidence → reward 0; fresh RL updates → compare plan probabilities and replay.

In [ ]:
agent_result = collect_chapter_terminal(terminal_run)
display(JSON(agent_result, expanded=True))

**What you demonstrated:** Use traces to identify the problem, a verifier to score behavior, and fresh tests to measure a change. Choose between skill, tool, policy, prompt, or model changes based on the evidence.

Exit the TUI with `/exit` when done.